
Mock JWST NIRSpec G395M spectrum of a z=7 star-forming galaxy

High-redshift star-forming galaxy with strong rest-frame UV and optical
emission lines redshifted into the JWST NIRSpec G395M window (2.9–5.1 μm).
Lines include Lyα, CIV, HeII, CIII], [OII], Hβ, [OIII], and Hα, each
annotated with rest wavelength and vacuum-frame position.

Demonstrates redshift-induced line migration and relative strength in
the high-z star-forming regime, useful for planning JWST observations
at z ≳ 6.


In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # suppress XLA/PjRt C++ INFO+WARNING logs

import warnings

import jax
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.analysis.plotting import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")

# ============================================================================
# Model Setup
# ============================================================================
# Using bare-stellar SSP (fsps_prsc_miles_chabrier) with Cue nebular backend.
# Build the model without observation= to access the full SED for resampling.

ssp = tengri.load_ssp("fsps_prsc_miles_chabrier")

model = tengri.SEDModel.build(
    ssp,
    sfh={
        "type": "dpl",
        "all_params": tengri.FIXED,
        "tau_gyr": 0.05,
        "log_total_mass": 10.0,
        "alpha": 3.5,
        "beta": 2.2,
    },
    dust={"type": "two_component", "all_params": tengri.FIXED, "tau_diff": 0.08, "tau_bc": 0.15},
    neb={"type": "cue", "all_params": tengri.FIXED, "logZ_gas": -1.0, "logU": -2.0},
    redshift=tengri.Fixed(7.0),
)

p = dict(model.spec.sample(jax.random.PRNGKey(0)))

# ============================================================================
# Redshift and Observed-Frame Setup
# ============================================================================
z = 7.0
rest_to_obs = 1.0 + z

# JWST NIRSpec G395M: 2.9–5.1 μm observed-frame
wave_obs_um = np.linspace(2.9, 5.1, 4000)
wave_obs_aa = wave_obs_um * 1e4  # Convert to Ångströms

# ============================================================================
# Predict Spectrum (Rest-frame SED, no observation)
# ============================================================================
out = model.predict(p)
wave_rest = np.asarray(model.wavelengths)  # Rest-frame Å
sed_rest = np.asarray(out.rest_sed())  # Rest-frame L_ν (erg/s/Hz)

# Map to observed frame via linear interpolation
# Observed wavelength maps back to rest via: λ_rest = λ_obs / (1+z)
wave_rest_from_obs = wave_obs_aa / rest_to_obs
# Interpolate rest SED at these rest wavelengths
sed_obs = np.interp(wave_rest_from_obs, wave_rest, sed_rest, left=0, right=0)

# Convert to f_λ for intuitive display
C_AA_PER_S = 2.998e18
f_lam_obs = sed_obs * C_AA_PER_S / (wave_obs_aa**2)

# ============================================================================
# Emission Lines (Rest-frame Vacuum Wavelengths)
# ============================================================================
# Rest-frame vacuum wavelengths (Å) and line names
# Source: NIST Atomic Line List, vacuum wavelengths
EMISSION_LINES = [
    (1215.67, r"Ly$\alpha$"),
    (1549.48, r"C\,IV"),
    (1640.42, r"He\,II"),
    (1908.73, r"C\,III]"),
    (3726.03, r"[O\,II]"),
    (4861.33, r"H$\beta$"),
    (5006.84, r"[O\,III]"),
    (6562.80, r"H$\alpha$"),
]

# ============================================================================
# Plot
# ============================================================================
fig, ax = plt.subplots(figsize=(10.5, 5.8))

# Spectrum
ax.plot(wave_obs_um, f_lam_obs, color="0.15", lw=1.2)

# Emission line indicators
ymin = float(np.percentile(f_lam_obs[f_lam_obs > 0], 15))
ymax = float(np.max(f_lam_obs[f_lam_obs > 0])) * 1.4

for lam_rest, line_name in EMISSION_LINES:
    # Shift to observed frame
    lam_obs = lam_rest * rest_to_obs
    lam_obs_um = lam_obs / 1e4

    # Only annotate if in the NIRSpec window
    if 2.9 <= lam_obs_um <= 5.1:
        ax.axvline(lam_obs_um, color="C1", lw=0.5, alpha=0.5, linestyle="--")
        # Label with rest wavelength, staggering height for readability
        y_pos = ymax * (0.82 + 0.08 * (hash(lam_rest) % 3))
        ax.text(
            lam_obs_um,
            y_pos,
            f"{line_name}\n({lam_rest:.1f} Å)",
            fontsize=7,
            color="C1",
            ha="center",
            va="top",
            rotation=0,
            bbox=dict(
                boxstyle="round,pad=0.3",
                facecolor="white",
                edgecolor="C1",
                alpha=0.8,
                linewidth=0.5,
            ),
        )

ax.set_xlim(2.9, 5.1)
ax.set_ylim(ymin, ymax)
ax.set_xlabel(r"Observed wavelength [$\mu$m]")
ax.set_ylabel(r"$F_\lambda$ [arbitrary]")

plt.savefig("plot_jwst_nirspec_high_z_spectrum.png", dpi=150, bbox_inches="tight")